In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Cargar los datos
file_path = '/content/Bomba.xlsx'  # Ajusta la ruta si es necesario
data = pd.read_excel(file_path)

# Reemplazar valores "NAN" (cadena) por valores NaN (nulos reales)
data.replace("NAN", pd.NA, inplace=True)

# Verificar si hay valores nulos y eliminarlos de las columnas clave
columns_to_check = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_cleaned = data.dropna(subset=columns_to_check)

# Gráfica 1: Aceleración en los 3 ejes
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=data_cleaned.index, y=data_cleaned['ACEL X'], mode='lines', name='Aceleración X', line=dict(color='tomato')))
fig1.add_trace(go.Scatter(x=data_cleaned.index, y=data_cleaned['ACEL Y'], mode='lines', name='Aceleración Y', line=dict(color='dodgerblue')))
fig1.add_trace(go.Scatter(x=data_cleaned.index, y=data_cleaned['ACEL Z'], mode='lines', name='Aceleración Z', line=dict(color='forestgreen')))
fig1.update_layout(title='Aceleración en los 3 Ejes',
                   xaxis_title='Índice de Tiempo',
                   yaxis_title='Aceleración',
                   template='plotly_white')
fig1.show()

# Gráfica 2: Velocidad en los 3 ejes
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=data_cleaned.index, y=data_cleaned['VELOC X'], mode='lines', name='Velocidad X', line=dict(color='gold')))
fig2.add_trace(go.Scatter(x=data_cleaned.index, y=data_cleaned['VELOC Y'], mode='lines', name='Velocidad Y', line=dict(color='purple')))
fig2.add_trace(go.Scatter(x=data_cleaned.index, y=data_cleaned['VELOC Z'], mode='lines', name='Velocidad Z', line=dict(color='brown')))
fig2.update_layout(title='Velocidad en los 3 Ejes',
                   xaxis_title='Índice de Tiempo',
                   yaxis_title='Velocidad',
                   template='plotly_white')
fig2.show()

# Gráfica 3: Temperatura
fig3 = px.line(data_cleaned, x=data_cleaned.index, y='TEMPERATURA', title='Temperatura', labels={'x': 'Índice de Tiempo', 'y': 'Temperatura'})
fig3.update_traces(line_color='black')
fig3.update_layout(template='plotly_white')
fig3.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.manifold import TSNE
import plotly.express as px

# Cargar los datos
file_path = '/content/Bomba.xlsx'  # Ajusta la ruta si es necesario
data = pd.read_excel(file_path)

# Reemplazar valores "NAN" (cadena) por valores NaN (nulos reales)
data.replace("NAN", pd.NA, inplace=True)

# Eliminar filas con valores nulos
columns_to_check = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_cleaned = data.dropna(subset=columns_to_check)

# Escalar las características (velocidad, aceleración, temperatura)
scaler = StandardScaler()
features = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_scaled = scaler.fit_transform(data_cleaned[features])

# Crear y ajustar el modelo One-Class SVM con kernel RBF
ocsvm = OneClassSVM(kernel='rbf', nu=0.1, gamma='scale')  # Ajusta nu según sea necesario
ocsvm.fit(data_scaled)

# Predecir si los puntos son outliers (1 = normal, -1 = outlier)
outliers = ocsvm.predict(data_scaled)
data_cleaned['outlier'] = np.where(outliers == -1, 1, 0)  # Etiqueta 1 para outliers, 0 para normales

# Aplicar t-SNE para reducir a 2 dimensiones
tsne = TSNE(n_components=2, random_state=42)
tsne_results = tsne.fit_transform(data_scaled)

# Agregar los resultados de t-SNE al DataFrame
data_cleaned['tSNE-1'] = tsne_results[:, 0]
data_cleaned['tSNE-2'] = tsne_results[:, 1]

# Crear el gráfico con Plotly
fig = px.scatter(data_cleaned, x='tSNE-1', y='tSNE-2',
                 color='outlier',  # Colorear según si es outlier o no
                 title='t-SNE para Velocidad, Aceleración y Temperatura',
                 labels={'tSNE-1': 'Componente 1', 'tSNE-2': 'Componente 2', 'outlier': 'Outlier'},
                 hover_data=['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA'])

# Mostrar el gráfico
fig.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.manifold import TSNE
import plotly.express as px

# Cargar los datos
file_path = '/content/Bomba.xlsx'  # Ajusta la ruta si es necesario
data = pd.read_excel(file_path)

# Reemplazar valores "NAN" (cadena) por valores NaN (nulos reales)
data.replace("NAN", pd.NA, inplace=True)

# Eliminar filas con valores nulos
columns_to_check = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_cleaned = data.dropna(subset=columns_to_check)

# Escalar las características (velocidad, aceleración, temperatura)
scaler = StandardScaler()
features = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_scaled = scaler.fit_transform(data_cleaned[features])

# Crear y ajustar el modelo One-Class SVM con kernel RBF
ocsvm = OneClassSVM(kernel='rbf', nu=0.1, gamma='scale')  # Ajusta nu según sea necesario
ocsvm.fit(data_scaled)

# Predecir si los puntos son outliers (1 = normal, -1 = outlier)
outliers = ocsvm.predict(data_scaled)
data_cleaned['outlier'] = np.where(outliers == -1, 1, 0)  # Etiqueta 1 para outliers, 0 para normales

# Aplicar t-SNE para reducir a 2 dimensiones
tsne = TSNE(n_components=2, random_state=42)
tsne_results = tsne.fit_transform(data_scaled)

# Agregar los resultados de t-SNE al DataFrame
data_cleaned['tSNE-1'] = tsne_results[:, 0]
data_cleaned['tSNE-2'] = tsne_results[:, 1]

# Crear el gráfico con Plotly
fig = px.scatter(data_cleaned, x='tSNE-1', y='tSNE-2',
                 color='TEMPERATURA',  # Colorear en función de la temperatura
                 symbol='outlier',  # Diferenciar outliers por símbolo
                 title='t-SNE para Velocidad, Aceleración y Temperatura',
                 labels={'tSNE-1': 'Componente 1', 'tSNE-2': 'Componente 2', 'TEMPERATURA': 'Temperatura', 'outlier': 'Outlier'},
                 hover_data=['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA'])

# Mostrar el gráfico
fig.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.manifold import TSNE
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import plotly.express as px

# Cargar los datos
file_path = '/content/Bomba.xlsx'  # Ajusta la ruta si es necesario
data = pd.read_excel(file_path)

# Reemplazar valores "NAN" (cadena) por valores NaN (nulos reales)
data.replace("NAN", pd.NA, inplace=True)

# Eliminar filas con valores nulos
columns_to_check = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_cleaned = data.dropna(subset=columns_to_check)

# Simular etiquetas verdaderas (solo para demostración; reemplazar con datos reales si están disponibles)
data_cleaned['true_outlier'] = np.random.choice([0, 1], size=len(data_cleaned), p=[0.9, 0.1])

# Dividir en conjunto de entrenamiento y validación
train_data, val_data = train_test_split(data_cleaned, test_size=0.3, stratify=data_cleaned['true_outlier'], random_state=42)

# Escalar las características (velocidad, aceleración, temperatura)
scaler = StandardScaler()
features = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
train_scaled = scaler.fit_transform(train_data[features])
val_scaled = scaler.transform(val_data[features])

# Crear y ajustar el modelo One-Class SVM con kernel RBF
ocsvm = OneClassSVM(kernel='rbf', nu=0.1, gamma='scale')  # Ajusta nu según sea necesario
ocsvm.fit(train_scaled)

# Predecir sobre el conjunto de validación
val_preds = ocsvm.predict(val_scaled)
val_data['outlier'] = np.where(val_preds == -1, 1, 0)

# Calcular métricas
print("\nReporte de Clasificación sobre el conjunto de validación:")
print(classification_report(val_data['true_outlier'], val_data['outlier'], target_names=['Normal', 'Outlier']))
print("\nMatriz de Confusión:")
print(confusion_matrix(val_data['true_outlier'], val_data['outlier']))

# Aplicar t-SNE para reducir a 2 dimensiones
tsne = TSNE(n_components=2, random_state=42)
tsne_results = tsne.fit_transform(val_scaled)
val_data['tSNE-1'] = tsne_results[:, 0]
val_data['tSNE-2'] = tsne_results[:, 1]

# Crear una columna de color solo para los outliers
val_data['color_temp'] = val_data.apply(lambda row: row['TEMPERATURA'] if row['outlier'] == 1 else None, axis=1)

# Crear el gráfico con Plotly
fig = px.scatter(val_data, x='tSNE-1', y='tSNE-2',
                 color='color_temp',  # Solo colorear los outliers por temperatura
                 symbol='outlier',  # Diferenciar outliers por símbolo
                 title='t-SNE para Validación - Solo outliers coloreados',
                 labels={'tSNE-1': 'Componente 1', 'tSNE-2': 'Componente 2', 'color_temp': 'Temperatura (solo outliers)', 'outlier': 'Outlier'},
                 hover_data=['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA'])

# Mostrar el gráfico
fig.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.manifold import TSNE
import plotly.express as px

# Cargar los datos
file_path = '/content/Bomba.xlsx'  # Ajusta la ruta si es necesario
data = pd.read_excel(file_path)

# Reemplazar valores "NAN" (cadena) por valores NaN (nulos reales)
data.replace("NAN", pd.NA, inplace=True)

# Eliminar filas con valores nulos
columns_to_check = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_cleaned = data.dropna(subset=columns_to_check)

# Escalar las características (velocidad, aceleración, temperatura)
scaler = StandardScaler()
features = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_scaled = scaler.fit_transform(data_cleaned[features])

# Crear y ajustar el modelo One-Class SVM con kernel RBF
ocsvm = OneClassSVM(kernel='rbf', nu=0.1, gamma='scale')  # Ajusta nu según sea necesario
ocsvm.fit(data_scaled)

# Predecir si los puntos son outliers (1 = normal, -1 = outlier)
outliers = ocsvm.predict(data_scaled)
data_cleaned['outlier'] = np.where(outliers == -1, 1, 0)  # Etiqueta 1 para outliers, 0 para normales

# Aplicar t-SNE para reducir a 2 dimensiones
tsne = TSNE(n_components=2, random_state=42)
tsne_results = tsne.fit_transform(data_scaled)

# Agregar los resultados de t-SNE al DataFrame
data_cleaned['tSNE-1'] = tsne_results[:, 0]
data_cleaned['tSNE-2'] = tsne_results[:, 1]

# Crear una columna para colorear solo los outliers con la temperatura
data_cleaned['color_temp'] = np.where(data_cleaned['outlier'] == 1, data_cleaned['TEMPERATURA'], np.nan)

# Crear el gráfico con Plotly
fig = px.scatter(data_cleaned, x='tSNE-1', y='tSNE-2',
                 color='color_temp',  # Solo los outliers tienen color según la temperatura
                 color_continuous_scale='plasma',  # Ajusta la escala de color
                 symbol='outlier',
                 title='t-SNE para Velocidad, Aceleración y Temperatura (Outliers)',
                 labels={'tSNE-1': 'Componente 1', 'tSNE-2': 'Componente 2', 'color_temp': 'Temperatura', 'outlier': 'Outlier'},
                 hover_data=['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA'])

# Mostrar el gráfico
fig.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.manifold import TSNE
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
import plotly.express as px

# Cargar los datos
file_path = '/content/Bomba.xlsx'  # Ajusta la ruta si es necesario
data = pd.read_excel(file_path)

# Reemplazar valores "NAN" (cadena) por valores NaN (nulos reales)
data.replace("NAN", pd.NA, inplace=True)

# Eliminar filas con valores nulos
columns_to_check = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_cleaned = data.dropna(subset=columns_to_check)

# Simular etiquetas verdaderas (solo para demostración; reemplazar con datos reales si están disponibles)
data_cleaned['true_outlier'] = np.random.choice([0, 1], size=len(data_cleaned), p=[0.9, 0.1])

# Dividir en conjunto de entrenamiento y validación
train_data, val_data = train_test_split(data_cleaned, test_size=0.3, stratify=data_cleaned['true_outlier'], random_state=42)

# Escalar las características
scaler = StandardScaler()
features = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
train_scaled = scaler.fit_transform(train_data[features])
val_scaled = scaler.transform(val_data[features])

# Definir combinaciones de hiperparámetros
models = {
    "Modelo 1 (nu=0.05, gamma=0.01)": OneClassSVM(kernel='rbf', nu=0.05, gamma=0.01),
    "Modelo 2 (nu=0.1, gamma='scale')": OneClassSVM(kernel='rbf', nu=0.1, gamma='scale'),
    "Modelo 3 (nu=0.15, gamma=0.1)": OneClassSVM(kernel='rbf', nu=0.15, gamma=0.1)
}

# Evaluar cada modelo
for name, model in models.items():
    print(f"\n🔍 Evaluando: {name}")
    model.fit(train_scaled)

    # Predicción y etiquetado
    val_preds = model.predict(val_scaled)
    val_data['outlier'] = np.where(val_preds == -1, 1, 0)

    # Métricas
    print(classification_report(val_data['true_outlier'], val_data['outlier'], target_names=['Normal', 'Outlier']))
    print("Matriz de confusión:")
    print(confusion_matrix(val_data['true_outlier'], val_data['outlier']))
    f1 = f1_score(val_data['true_outlier'], val_data['outlier'])
    print(f"F1-score: {f1:.4f}")

    # t-SNE
    tsne = TSNE(n_components=2, random_state=42)
    tsne_results = tsne.fit_transform(val_scaled)
    val_data['tSNE-1'] = tsne_results[:, 0]
    val_data['tSNE-2'] = tsne_results[:, 1]

    # Colorear solo los outliers
    val_data['color_temp'] = val_data.apply(lambda row: row['TEMPERATURA'] if row['outlier'] == 1 else None, axis=1)

    # Gráfico
    fig = px.scatter(val_data, x='tSNE-1', y='tSNE-2',
                     color='color_temp',
                     color_continuous_scale='plasma',
                     symbol='outlier',
                     title=f'{name} - t-SNE (Outliers coloreados por temperatura)',
                     labels={'tSNE-1': 'Componente 1', 'tSNE-2': 'Componente 2', 'color_temp': 'Temperatura', 'outlier': 'Outlier'},
                     hover_data=['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA'])
    fig.show()


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

# Cargar los datos
file_path = '/content/Bomba.xlsx'
data = pd.read_excel(file_path)

# Reemplazar valores "NAN" (cadena) por valores NaN (nulos reales)
data.replace("NAN", pd.NA, inplace=True)

# Verificar si hay valores nulos y eliminarlos de las columnas
columns_to_check = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_cleaned = data.dropna(subset=columns_to_check)

# Calcular la matriz de correlación
correlation_matrix = data_cleaned[columns_to_check].corr()

# --- Matriz de correlación interactiva (Plotly) ---
fig_corr = px.imshow(correlation_matrix,
                     labels=dict(x="Variables", y="Variables", color="Correlación"),
                     x=columns_to_check,
                     y=columns_to_check,
                     color_continuous_scale="Viridis")  # Cambiar "Viridis" por otro de la paleta de colores que se requiera
fig_corr.update_layout(title="Matriz de Correlación Interactiva", title_x=0.5)
fig_corr.show()

# Función para detectar anomalías con Z-Score
def detect_anomalies(series, threshold=3):
    mean = np.mean(series)
    std = np.std(series)
    z_scores = (series - mean) / std
    return z_scores.abs() > threshold

# Detectar anomalías para cada columna
anomalies = {}
for col in columns_to_check:
    anomalies[col] = detect_anomalies(data_cleaned[col])

# Agregar columnas de anomalías al DataFrame
for col in columns_to_check:
    data_cleaned[f'Anomalía_{col}'] = anomalies[col]

# Gráfica 1: Aceleración en los 3 ejes con anomalías
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=data_cleaned.index, y=data_cleaned['ACEL X'], mode='lines', name='Aceleración X', line=dict(color='tomato')))
fig1.add_trace(go.Scatter(x=data_cleaned.index, y=data_cleaned['ACEL Y'], mode='lines', name='Aceleración Y', line=dict(color='dodgerblue')))
fig1.add_trace(go.Scatter(x=data_cleaned.index, y=data_cleaned['ACEL Z'], mode='lines', name='Aceleración Z', line=dict(color='forestgreen')))
# Añadir anomalías como puntos
for col, color in zip(['ACEL X', 'ACEL Y', 'ACEL Z'], ['tomato', 'dodgerblue', 'forestgreen']):
    fig1.add_trace(go.Scatter(x=data_cleaned.index[data_cleaned[f'Anomalía_{col}']],
                              y=data_cleaned[col][data_cleaned[f'Anomalía_{col}']],
                              mode='markers', name=f'Anomalía {col}', marker=dict(color=color, size=8, symbol='x')))
fig1.update_layout(title='Aceleración en los 3 Ejes (Con Anomalías)',
                   xaxis_title='Índice de Tiempo',
                   yaxis_title='Aceleración',
                   template='plotly_white')
fig1.show()

# Gráfica 2: Velocidad en los 3 ejes con anomalías
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=data_cleaned.index, y=data_cleaned['VELOC X'], mode='lines', name='Velocidad X', line=dict(color='gold')))
fig2.add_trace(go.Scatter(x=data_cleaned.index, y=data_cleaned['VELOC Y'], mode='lines', name='Velocidad Y', line=dict(color='purple')))
fig2.add_trace(go.Scatter(x=data_cleaned.index, y=data_cleaned['VELOC Z'], mode='lines', name='Velocidad Z', line=dict(color='brown')))
# Añadir anomalías como puntos
for col, color in zip(['VELOC X', 'VELOC Y', 'VELOC Z'], ['gold', 'purple', 'brown']):
    fig2.add_trace(go.Scatter(x=data_cleaned.index[data_cleaned[f'Anomalía_{col}']],
                              y=data_cleaned[col][data_cleaned[f'Anomalía_{col}']],
                              mode='markers', name=f'Anomalía {col}', marker=dict(color=color, size=8, symbol='x')))
fig2.update_layout(title='Velocidad en los 3 Ejes (Con Anomalías)',
                   xaxis_title='Índice de Tiempo',
                   yaxis_title='Velocidad',
                   template='plotly_white')
fig2.show()

# Gráfica 3: Temperatura con anomalías
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=data_cleaned.index, y=data_cleaned['TEMPERATURA'], mode='lines', name='Temperatura', line=dict(color='black')))
# Añadir anomalías como puntos
fig3.add_trace(go.Scatter(x=data_cleaned.index[data_cleaned['Anomalía_TEMPERATURA']],
                          y=data_cleaned['TEMPERATURA'][data_cleaned['Anomalía_TEMPERATURA']],
                          mode='markers', name='Anomalía Temperatura', marker=dict(color='red', size=8, symbol='x')))
fig3.update_layout(title='Temperatura (Con Anomalías)',
                   xaxis_title='Índice de Tiempo',
                   yaxis_title='Temperatura',
                   template='plotly_white')
fig3.show()


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
import matplotlib.pyplot as plt

# --- Preparar los datos ---
# Cargar los datos
file_path = '/content/Bomba.xlsx'
data = pd.read_excel(file_path)

# Reemplazar valores "NAN" (cadena) por valores NaN (nulos reales)
data.replace("NAN", pd.NA, inplace=True)

# Verificar si hay valores nulos y eliminarlos de las columnas
columns_to_check = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_cleaned = data.dropna(subset=columns_to_check)

# Detectar anomalías con Z-Score
def detect_anomalies(series, threshold=3):
    mean = np.mean(series)
    std = np.std(series)
    z_scores = (series - mean) / std
    return z_scores.abs() > threshold

# Agregar columnas de anomalías
for col in columns_to_check:
    data_cleaned[f'Anomalía_{col}'] = detect_anomalies(data_cleaned[col])

# Separar características y etiquetas
X = data_cleaned[['TEMPERATURA', 'VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z']]
y = data_cleaned[['Anomalía_TEMPERATURA', 'Anomalía_VELOC X', 'Anomalía_VELOC Y',
                  'Anomalía_VELOC Z', 'Anomalía_ACEL X', 'Anomalía_ACEL Y', 'Anomalía_ACEL Z']]

# Escalar las características
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# --- Construcción del modelo ---
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(y_train.shape[1], activation='sigmoid')  # Salida para cada tipo de anomalía
])

# Compilar el modelo
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Entrenar el modelo
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test), verbose=1)

# --- Predicciones ---
y_pred = model.predict(X_test)
y_pred_binary = (y_pred > 0.5).astype(int)

# Reconstruir el DataFrame con las predicciones
X_test_df = pd.DataFrame(scaler.inverse_transform(X_test), columns=X.columns)
y_test_df = pd.DataFrame(y_test, columns=y.columns)
y_pred_df = pd.DataFrame(y_pred_binary, columns=y.columns)

# Añadir predicciones, probabilidades y datos reales
results_df = X_test_df.copy()
for col, prob_col in zip(y.columns, y_pred.T):
    results_df[f'Predicción_{col}'] = y_pred_df[col]
    results_df[f'Probabilidad_{col}'] = prob_col
    results_df[f'Real_{col}'] = y_test_df[col]

# --- Graficar las anomalías y predicciones ---
def plot_all_variables():
    for variable, real_label, pred_label, prob_label in zip(
        ['TEMPERATURA', 'VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z'],
        ['Real_Anomalía_TEMPERATURA', 'Real_Anomalía_VELOC X', 'Real_Anomalía_VELOC Y',
         'Real_Anomalía_VELOC Z', 'Real_Anomalía_ACEL X', 'Real_Anomalía_ACEL Y', 'Real_Anomalía_ACEL Z'],
        ['Predicción_Anomalía_TEMPERATURA', 'Predicción_Anomalía_VELOC X', 'Predicción_Anomalía_VELOC Y',
         'Predicción_Anomalía_VELOC Z', 'Predicción_Anomalía_ACEL X', 'Predicción_Anomalía_ACEL Y',
         'Predicción_Anomalía_ACEL Z'],
        ['Probabilidad_Anomalía_TEMPERATURA', 'Probabilidad_Anomalía_VELOC X',
         'Probabilidad_Anomalía_VELOC Y', 'Probabilidad_Anomalía_VELOC Z', 'Probabilidad_Anomalía_ACEL X',
         'Probabilidad_Anomalía_ACEL Y', 'Probabilidad_Anomalía_ACEL Z']
    ):
        fig = go.Figure()

        # Línea principal de la variable
        fig.add_trace(go.Scatter(x=results_df.index, y=results_df[variable],
                                 mode='lines', name=f'{variable}', line=dict(color='black')))

        # Fallos reales
        fig.add_trace(go.Scatter(
            x=results_df.index[results_df[real_label] == 1],
            y=results_df[variable][results_df[real_label] == 1],
            mode='markers',
            name=f'Fallo real ({variable})',
            marker=dict(color='blue', size=8, symbol='circle'),
            hoverinfo='text',
            text=[f"Valor: {v:.2f}<br>Fallo real" for v in results_df[variable][results_df[real_label] == 1]]
        ))

        # Predicciones de fallo
        fig.add_trace(go.Scatter(
            x=results_df.index[results_df[pred_label] == 1],
            y=results_df[variable][results_df[pred_label] == 1],
            mode='markers',
            name=f'Predicción de fallo ({variable})',
            marker=dict(color='red', size=8, symbol='x'),
            hoverinfo='text',
            text=[f"Valor: {v:.2f}<br>Probabilidad: {p:.2f}"
                  for v, p in zip(results_df[variable][results_df[pred_label] == 1],
                                  results_df[prob_label][results_df[pred_label] == 1])]
        ))

        fig.update_layout(title=f'{variable} con Predicciones y Fallos',
                          xaxis_title='Índice de Tiempo',
                          yaxis_title=f'{variable}',
                          template='plotly_white')
        fig.show()

# Ejecutar las gráficas
plot_all_variables()

# --- Graficar las métricas del modelo ---
# Precisión
plt.figure(figsize=(12, 6))
plt.plot(history.history['accuracy'], label='Precisión de entrenamiento')
plt.plot(history.history['val_accuracy'], label='Precisión de validación')
plt.title('Curva de Precisión del Modelo')
plt.xlabel('Épocas')
plt.ylabel('Precisión')
plt.legend()
plt.grid(True)
plt.show()

# Pérdida
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Pérdida de entrenamiento')
plt.plot(history.history['val_loss'], label='Pérdida de validación')
plt.title('Curva de Pérdida del Modelo')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report

# 1. Cargar y limpiar los datos
file_path = '/content/Bomba.xlsx'
data = pd.read_excel(file_path)
data.replace("NAN", pd.NA, inplace=True)

columns_to_check = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_cleaned = data.dropna(subset=columns_to_check).copy()

features = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']

# 2. Escalado de características
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data_cleaned[features])

# 3. Primer modelo (el de referencia)
modelo_base = OneClassSVM(kernel='rbf', nu=0.1, gamma='scale')
modelo_base.fit(X_scaled)
pred_base = np.where(modelo_base.predict(X_scaled) == -1, 1, 0)  # 1 = outlier

# Guardar como "etiqueta real"
data_cleaned['etiqueta_real'] = pred_base

# 4. Nuevo modelo con parámetros distintos (por ejemplo: más conservador)
modelo_nuevo = OneClassSVM(kernel='rbf', nu=0.05, gamma='scale')
modelo_nuevo.fit(X_scaled)
pred_nuevo = np.where(modelo_nuevo.predict(X_scaled) == -1, 1, 0)

# 5. Evaluar el nuevo modelo contra la etiqueta del modelo base
print("Evaluación del nuevo modelo contra el modelo base (como referencia):\n")
print(classification_report(data_cleaned['etiqueta_real'], pred_nuevo))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, recall_score

# Cargar datos
file_path = '/content/Bomba.xlsx'
data = pd.read_excel(file_path)
data.replace("NAN", pd.NA, inplace=True)

# Limpiar
columns_to_check = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
data_cleaned = data.dropna(subset=columns_to_check).copy()

# Escalar
features = ['VELOC X', 'VELOC Y', 'VELOC Z', 'ACEL X', 'ACEL Y', 'ACEL Z', 'TEMPERATURA']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data_cleaned[features])

# Modelo base (nu=0.1, gamma='scale') como etiquetas reales
modelo_base = OneClassSVM(kernel='rbf', nu=0.1, gamma='scale')
modelo_base.fit(X_scaled)
y_true = np.where(modelo_base.predict(X_scaled) == -1, 1, 0)

# Configuraciones a probar
configs = [
    {"nu": 0.05, "gamma": "scale"},
    {"nu": 0.15, "gamma": "scale"},
    {"nu": 0.1,  "gamma": 0.001},
    {"nu": 0.1,  "gamma": 1},
    {"nu": 0.2,  "gamma": "scale"},
    {"nu": 0.05, "gamma": 0.01}
]

# Evaluar cada configuración
resultados = []

for i, config in enumerate(configs):
    modelo = OneClassSVM(kernel='rbf', nu=config['nu'], gamma=config['gamma'])
    modelo.fit(X_scaled)
    y_pred = np.where(modelo.predict(X_scaled) == -1, 1, 0)

    report = classification_report(y_true, y_pred, output_dict=True)
    recall = report['1']['recall']
    precision = report['1']['precision']
    f1 = report['1']['f1-score']

    resultados.append({
        "config": f"nu={config['nu']}, gamma={config['gamma']}",
        "recall": recall,
        "precision": precision,
        "f1-score": f1
    })

# Mostrar resultados ordenados por recall
resultados_ordenados = sorted(resultados, key=lambda x: x['recall'], reverse=True)

# Mostrar tabla bonita
import pandas as pd
df_resultados = pd.DataFrame(resultados_ordenados)
print("\nResultados ordenados por RECALL:")
print(df_resultados.to_string(index=False))



Finalmente los gráficos representan las métricas de entrenamiento y validación del modelo de red neuronal.

Curva de Precisión del Modelo: Este gráfico muestra cómo varía la precisión del modelo en el conjunto de entrenamiento y en el conjunto de validación a lo largo de las épocas. La precisión mide el porcentaje de predicciones correctas hechas por el modelo. En este caso, las líneas muestran fluctuaciones significativas en ambas curvas (entrenamiento y validación). Estas oscilaciones podrían ser indicativas de un modelo que no se está entrenando de manera consistente, posiblemente debido a datos ruidosos o un modelo que requiere ajustes en los hiperparámetros, como la tasa de aprendizaje o la arquitectura de la red.

Curva de Pérdida del Modelo: Este gráfico muestra cómo varía la función de pérdida (binary cross-entropy) del modelo durante el entrenamiento y la validación. La pérdida es una medida de qué tan mal están las predicciones del modelo con respecto a las etiquetas reales. En este gráfico, se observa que la pérdida del conjunto de entrenamiento disminuye de manera constante, lo que indica que el modelo está aprendiendo a ajustarse mejor a los datos. Por otro lado, la pérdida de validación también disminuye pero tiene fluctuaciones, lo que sugiere que el modelo está aprendiendo pero podría estar sobreajustándose en algunas épocas o lidiando con un conjunto de validación ruidoso.

En resumen, estos gráficos indican que el modelo está aprendiendo, pero las fluctuaciones en las métricas de validación podrían significar que es necesario realizar ajustes adicionales, como aumentar el conjunto de datos, ajustar el dropout para evitar el sobreajuste o mejorar la calidad de los datos de entrada.